# Journal Search-and-Summarize — Colab / Kaggle / Binder companion

This notebook mirrors the local `uv` project from the course's
[Search and Summarize Your Own Journal](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/journal-search-summarize)
lesson, adapted to run in a hosted notebook with no local files: embed a folder's
worth of dated journal entries locally with `sentence-transformers`, search them
semantically with NumPy, and summarize a date range with a free-tier LLM that
cites the date behind every claim.

The real version reads your own `YYYY-MM-DD.md` files from `data/journal/` on disk
(see the [local example project](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/journal-search-summarize));
this hosted notebook just embeds the same twelve sample entries directly as
strings, so everything runs with zero setup. See the
[lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/journal-search-summarize)
for the full walkthrough.

**Step 0 — install dependencies:**

In [ ]:
!pip install -q sentence-transformers numpy openai python-dotenv

## Step 1: Sample journal

The same twelve entries shipped with the local example (`data/journal/`), each
keyed by its `YYYY-MM-DD` filename. They cover about two and a half weeks of
recurring themes -- work on a data pipeline, a side-project finance tracker, a
trip to Chefchaouen being planned with friends, and a family birthday -- so
specific queries have a clear right answer.

In [ ]:
JOURNAL = {
    "2026-07-06": """Back to work after the weekend. Our data team kicked off the
quarterly pipeline migration today -- my main task for the next few weeks.

After work I started sketching a small personal project: a finance tracker CLI
that reads my monthly expenses and prints a summary. Nothing fancy, just enough
to stop doing the math by hand.

Short run in the evening, 3 km around the neighborhood.""",
    "2026-07-07": """Long planning meeting in the morning; nothing decided, lots of
diagrams.

During lunch, Yasmine and Omar brought up doing a short trip to Chefchaouen at
the end of August. We talked about dates, the blue medina, and whether to take
the bus or the train. No bookings yet, but the idea is on the table.""",
    "2026-07-08": """Spent most of the day debugging an ETL bug in the staging
pipeline -- turned out to be a timezone offset in the daily aggregation step.
Fixed after lunch.

Started the finance tracker project for real: created the repo and wrote the
first module for recording a single expense (amount, category, date).""",
    "2026-07-09": """Good 5k run after work, first time this week I felt fast.

Called mom in the evening. Her birthday is on July 18 and we're planning a
family dinner at home -- she wants the usual: her cooking, all of us at the
table, no restaurant.""",
    "2026-07-10": """Wrapped up the pipeline report and sent it to the team. Feels
good to close out the week.

Trip planning: compared a few riads in Chefchaouen with Omar. Shortlist is down
to two -- one in the medina near the main square, one quieter up the hill.""",
    "2026-07-14": """Morning run before work, 5 km. The weather is getting hot, so
early runs are the way to go.

Finance tracker: added CSV import so I can load the bank export instead of
typing every expense by hand. Imported June's data and the totals matched,
which was satisfying.""",
    "2026-07-15": """Long debugging session on the ETL again -- the same staging
pipeline, a new failure in the deduplication step. Fixed it, but it ate the
whole day.

Skipped the gym. Sometimes a rest day wins.""",
    "2026-07-16": """Trip is getting real: Yasmine confirmed she and Omar can do
August 22-25 for Chefchaouen. I checked train times from Casablanca -- the
direct route is a few hours, and the buses are cheaper but slower. We'll decide
together this weekend.""",
    "2026-07-18": """Mom's birthday. Family dinner at home as planned -- her
cooking, everyone around the table: my parents, my two brothers, my sister and
her kids. Small gift, big cake, a really good evening.""",
    "2026-07-20": """Shipped the ETL fix to production this morning -- the staging
pipeline has been stable all day.

Started outlining the August sprint with the team: two more migration tasks and
a monitoring dashboard. Also kicked off planning the week's meals to avoid the
takeout spiral.""",
    "2026-07-21": """Morning run again, 5 km, felt great.

Finance tracker reached v0.2: added a monthly summary command that groups
expenses by category and prints totals. It still can't save my money, but at
least it shows where it goes.""",
    "2026-07-22": """Booked the riad in Chefchaouen for August 22-24 -- the one in
the medina near the main square. Two nights, three of us. Train tickets next,
then it's just packing.

Work was quiet: code review and a short refactor of the report module.""",
}

print(f"Loaded {len(JOURNAL)} journal entries: {sorted(JOURNAL)}")

## Step 2: Embed the journal locally

Same model as `main.py`'s `index` command: `all-MiniLM-L6-v2`, run entirely on
CPU, no API key needed. The date is prefixed onto each entry so it's part of
what gets embedded, matching the local version. Since this notebook has no
`index.npy`/`chunks.json` files to persist to, the embeddings just stay in
memory.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

dates = sorted(JOURNAL)
texts = [f"[{d}] {JOURNAL[d]}" for d in dates]

print(f"Embedding {len(texts)} entries with {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(texts, normalize_embeddings=True)

print(f"Embedded {embeddings.shape[0]} entries ({embeddings.shape[1]}-dim)")

## Step 3: Search semantically

Same retrieval as `main.py`'s `search` command: embed the question with the
same model, then rank every entry by cosine similarity -- which, because every
embedding is already normalized to length 1, collapses to a single matrix-vector
dot product.

In [ ]:
def search(query: str, top_k: int = 5) -> list[dict]:
    """Returns the top_k entries most similar to `query`, highest first."""
    query_vector = model.encode([query], normalize_embeddings=True)[0]
    similarities = embeddings @ query_vector
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [
        {"date": dates[i], "score": float(similarities[i]), "text": JOURNAL[dates[i]]}
        for i in top_indices
    ]


results = search("when did I last mention planning a trip?")
for r in results:
    snippet = r["text"].replace("\n", " ")[:100]
    print(f"{r['score']:.3f}  [{r['date']}]  {snippet}...")

## Step 4: Get a free-tier LLM API key

Search is fully local. Only the summarize step calls a hosted model. Pick any
provider from the table in the [lesson's Setup section](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/journal-search-summarize#get-a-free-llm-api-key) --
GitHub Models is the suggested default since it needs no separate signup. The
key is entered with `getpass` so it never gets typed into a visible cell or
saved into this notebook's output.

In [ ]:
import os
from getpass import getpass

# Pick a provider: github (default), gemini, groq, mistral, cerebras, openrouter.
LLM_PROVIDER = "github"

KEY_NAMES = {
    "github": "GITHUB_TOKEN",
    "gemini": "GOOGLE_API_KEY",
    "groq": "GROQ_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "cerebras": "CEREBRAS_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
}

key_name = KEY_NAMES[LLM_PROVIDER]
os.environ[key_name] = getpass(f"Enter your free-tier {LLM_PROVIDER} API key ({key_name}): ")

## Step 5: Summarize a date range

This mirrors `summarize` in `main.py`: pull every entry whose date falls in the
range, tag each with its date, and ask a free-tier LLM to write one bullet per
distinct event -- each bullet **beginning with the date it came from** -- using
only the journal text it was given. The `[YYYY-MM-DD]` citations are then
extracted from the response, so you can audit every sentence against its source
entry instead of trusting the summary blindly. Try a different range afterwards
-- for example `2026-07-06` to `2026-07-12` -- and notice how the bullets map
back to real entries.

In [ ]:
import re
from openai import OpenAI

SUMMARIZE_PROMPT = """Summarize this personal journal from {start} to {end}.

Below is the journal text for each day in that range, each tagged with its date
in brackets. Write a short summary of what happened during this period, one
bullet per distinct event or theme. Begin every bullet with the date it comes
from, in the same [YYYY-MM-DD] form. For example:

[2026-07-18] Dinner with the whole family for mom's birthday.

Use ONLY facts that appear in the journal text below. Do not invent events,
dates, or details, and do not add opinions. If the range contains no entries,
say so in one sentence.

Journal:
{context}

Summary:"""

PROVIDERS = {
    "github": ("https://models.github.ai/inference", "gpt-4o-mini"),
    "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-3.5-flash"),
    "groq": ("https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
    "mistral": ("https://api.mistral.ai/v1", "mistral-small-latest"),
    "cerebras": ("https://api.cerebras.ai/v1", "llama-3.3-70b"),
    "openrouter": ("https://openrouter.ai/api/v1", "meta-llama/llama-3.3-70b-instruct:free"),
}

base_url, model = PROVIDERS[LLM_PROVIDER]
client = OpenAI(api_key=os.environ[key_name], base_url=base_url)


def summarize(start: str, end: str) -> tuple[str, list[str]]:
    in_range = [d for d in dates if start <= d <= end]
    if not in_range:
        return f"No journal entries found between {start} and {end}.", []
    context = "\n\n".join(f"[{d}]\n{JOURNAL[d]}" for d in in_range)
    prompt = SUMMARIZE_PROMPT.format(start=start, end=end, context=context)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.choices[0].message.content
    cited = sorted({d for d in re.findall(r"\[(\d{4}-\d{2}-\d{2})\]", text) if start <= d <= end})
    return text, cited


summary_text, cited_dates = summarize("2026-07-13", "2026-07-22")
print(summary_text)
print()
print("Sources cited (audit trail):")
for d in cited_dates:
    print(f"  {d}  data/journal/{d}.md")